# Module 4 — Forward Kinematics using Denavit–Hartenberg Parameters
## Unit 4 — The Forward Kinematics Map
### Lesson 4.3 — Forward Kinematics in Code

*Physical AI Curriculum · hands-on notebook. Run **Kernel → Restart & Run All**.*

In [ ]:
import numpy as np
from functools import reduce
def se3_rotz(theta):
    c,s=np.cos(theta),np.sin(theta); return np.array([[c,-s,0,0],[s,c,0,0],[0,0,1,0],[0,0,0,1.0]])
def se3_trans(x,y,z):
    T=np.eye(4); T[0,3]=x;T[1,3]=y;T[2,3]=z; return T
def fk(factors):
    # build fresh; reduce over matrix products (no in-place aliasing)
    return reduce(lambda A,B: A@B, factors, np.eye(4))
L1,L2=0.4,0.3
lf=lambda th,L: se3_rotz(th)@se3_trans(L,0,0)
T=fk([lf(np.radians(30),L1),lf(np.radians(60),L2)])
print('numeric position',np.round(T[:3,3],3))

## Coding Exercise (§8)
Verify numeric FK vs the planar closed form, then derive the symbolic FK with SymPy.

In [ ]:
# YOUR CODE HERE
import numpy as np
def fk_planar(joints):
    phi=0.0;x=0.0;y=0.0
    for th,L in joints:
        phi+=th; x+=L*np.cos(phi); y+=L*np.sin(phi)
    return np.array([x,y])
assert np.allclose(T[:2,3],fk_planar([(np.radians(30),L1),(np.radians(60),L2)]),atol=1e-9)
import sympy as sp
t1,t2,l1,l2=sp.symbols('theta1 theta2 L1 L2',real=True)
def srotz(t): return sp.Matrix([[sp.cos(t),-sp.sin(t),0,0],[sp.sin(t),sp.cos(t),0,0],[0,0,1,0],[0,0,0,1]])
def strans(x,y,z): return sp.Matrix([[1,0,0,x],[0,1,0,y],[0,0,1,z],[0,0,0,1]])
Ts=(srotz(t1)*strans(l1,0,0))*(srotz(t2)*strans(l2,0,0))
px=sp.simplify(Ts[0,3]); py=sp.simplify(Ts[1,3])
expected_x=sp.simplify(l1*sp.cos(t1)+l2*sp.cos(t1+t2))
assert sp.simplify(px-expected_x)==0
print('symbolic position x =',px)

## Check your work

In [ ]:
import numpy as np
assert np.allclose(fk([lf(0,L1),lf(0,L2)])[:3,3],[0.7,0,0])
print('All checks passed.')